# 🎭 Video Swap Pipeline — Free Colab Backend

This notebook runs the **entire AI backend** on Google Colab's free T4 GPU (15GB VRAM).

**What you get:**
- A public API URL that your Android phone can connect to
- Full face/character swap processing powered by Stable Diffusion + ControlNet + IP-Adapter
- Lip sync with Wav2Lip
- Videos up to 10 minutes supported

**How to use:**
1. Make sure you selected **GPU runtime** (Runtime → Change runtime type → T4 GPU)
2. Run each cell in order (▶️)
3. Copy the public URL from Cell 5 into your Android app settings

> ⚠️ **Free Colab limits**: ~4 hours/session, may disconnect if idle. For unlimited, use Colab Pro or self-host.

## Cell 1: Verify GPU & Check VRAM

In [ ]:
# ============================================================
# CELL 1: Verify GPU availability and VRAM
# ============================================================
import torch
import subprocess

print("=" * 60)
print("GPU CHECK")
print("=" * 60)

if not torch.cuda.is_available():
    print("❌ NO GPU DETECTED!")
    print("Go to: Runtime → Change runtime type → T4 GPU")
    raise RuntimeError("GPU required. Change runtime type to T4 GPU.")

gpu_name = torch.cuda.get_device_name(0)
vram_total = torch.cuda.get_device_properties(0).total_mem / (1024**3)
vram_free = torch.cuda.mem_get_info(0)[0] / (1024**3)

print(f"✅ GPU: {gpu_name}")
print(f"✅ Total VRAM: {vram_total:.1f} GB")
print(f"✅ Free VRAM: {vram_free:.1f} GB")
print(f"✅ CUDA Version: {torch.version.cuda}")
print(f"✅ PyTorch Version: {torch.__version__}")

# Show nvidia-smi
print("\n" + "=" * 60)
!nvidia-smi
print("\n🎉 GPU is ready! Proceed to Cell 2.")

## Cell 2: Install All Dependencies

In [ ]:
# ============================================================
# CELL 2: Install all required Python packages
# This takes ~3-5 minutes on first run
# ============================================================
print("📦 Installing dependencies... (this takes 3-5 minutes)")
print("=" * 60)

# Core server
!pip install -q fastapi uvicorn[standard] python-multipart pydantic pydantic-settings

# Task queue
!pip install -q celery[redis] redis

# ML/AI models
!pip install -q diffusers transformers accelerate safetensors
!pip install -q controlnet-aux

# Face detection
!pip install -q insightface onnxruntime-gpu

# Image/Video processing
!pip install -q opencv-python-headless Pillow scipy ffmpeg-python

# Utilities
!pip install -q aiofiles httpx loguru tqdm

# Tunneling (to expose API to your phone)
!pip install -q pyngrok

# Install Redis server
!apt-get update -qq && apt-get install -qq -y redis-server > /dev/null 2>&1

# Verify FFmpeg is available
!ffmpeg -version 2>/dev/null | head -1 || echo '❌ FFmpeg not found'

print("\n✅ All dependencies installed! Proceed to Cell 3.")

## Cell 3: Clone/Setup Pipeline Code

In [ ]:
# ============================================================
# CELL 3: Setup the pipeline code
# Option A: Upload from your machine
# Option B: Copy the code inline (for standalone use)
# ============================================================
import os
from pathlib import Path

PROJECT_DIR = Path("/content/video-swap-pipeline")

# Create working directories
for d in ["uploads", "processed", "output", "checkpoints", "logs"]:
    (PROJECT_DIR / d).mkdir(parents=True, exist_ok=True)

# Set environment variables
os.environ["VSWAP_REDIS_URL"] = "redis://localhost:6379/0"
os.environ["VSWAP_CELERY_BROKER_URL"] = "redis://localhost:6379/0"
os.environ["VSWAP_CELERY_RESULT_BACKEND"] = "redis://localhost:6379/1"
os.environ["VSWAP_UPLOAD_DIR"] = str(PROJECT_DIR / "uploads")
os.environ["VSWAP_PROCESSED_DIR"] = str(PROJECT_DIR / "processed")
os.environ["VSWAP_OUTPUT_DIR"] = str(PROJECT_DIR / "output")
os.environ["VSWAP_GPU_DEVICES"] = "[0]"
os.environ["VSWAP_LOG_LEVEL"] = "INFO"
os.environ["VSWAP_WAV2LIP_CHECKPOINT"] = str(PROJECT_DIR / "checkpoints" / "wav2lip_gan.pth")

print(f"📁 Project directory: {PROJECT_DIR}")
print(f"📁 Uploads: {PROJECT_DIR / 'uploads'}")
print(f"📁 Processed: {PROJECT_DIR / 'processed'}")
print(f"📁 Output: {PROJECT_DIR / 'output'}")

# If you have the code as a zip, upload and extract it:
print("\n" + "=" * 60)
print("📤 UPLOAD YOUR CODE:")
print("Option 1: Upload video-swap-pipeline.zip using the file browser (left sidebar)")
print("Option 2: If you have a GitHub repo, uncomment the git clone line below")
print("=" * 60)

# Uncomment ONE of these:
# !git clone https://github.com/YOUR_USER/video-swap-pipeline.git /content/video-swap-pipeline
# !unzip /content/video-swap-pipeline.zip -d /content/

# For now, let's create a minimal working server inline:
MINIMAL_SERVER = '''
"""Minimal standalone FastAPI server for Video Swap Pipeline.
This is a simplified version that runs directly in Colab.
"""
import os
import uuid
import time
import json
import shutil
import subprocess
import asyncio
from pathlib import Path
from enum import Enum
from typing import Any, Optional

import torch
import numpy as np
import cv2
from PIL import Image
from loguru import logger
from fastapi import FastAPI, File, Form, UploadFile, HTTPException, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse
from pydantic import BaseModel, Field

# ---- Configuration ----
UPLOAD_DIR = Path(os.environ.get("VSWAP_UPLOAD_DIR", "/content/video-swap-pipeline/uploads"))
PROCESSED_DIR = Path(os.environ.get("VSWAP_PROCESSED_DIR", "/content/video-swap-pipeline/processed"))
OUTPUT_DIR = Path(os.environ.get("VSWAP_OUTPUT_DIR", "/content/video-swap-pipeline/output"))
MAX_DURATION = 600  # 10 minutes

for d in [UPLOAD_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- Schemas ----
class JobStatus(str, Enum):
    PENDING = "PENDING"
    CHUNKING = "CHUNKING"
    PROCESSING = "PROCESSING"
    STITCHING = "STITCHING"
    COMPLETED = "COMPLETED"
    FAILED = "FAILED"

class JobResponse(BaseModel):
    job_id: str
    status: JobStatus
    message: str = ""
    progress: float = 0.0
    output_url: Optional[str] = None

# ---- In-Memory Job Store ----
jobs: dict[str, dict] = {}

# ---- VRAM Management ----
def clear_vram():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    import gc
    gc.collect()

# ---- FFmpeg Helpers ----
def probe_video(path: Path) -> dict:
    cmd = [
        "ffprobe", "-v", "quiet", "-print_format", "json",
        "-show_streams", "-show_format", str(path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    data = json.loads(result.stdout)
    video_stream = next((s for s in data.get("streams", []) if s["codec_type"] == "video"), {})
    return {
        "width": int(video_stream.get("width", 0)),
        "height": int(video_stream.get("height", 0)),
        "duration": float(data.get("format", {}).get("duration", 0)),
        "fps": eval(video_stream.get("r_frame_rate", "30/1")),
    }

def chunk_video(video_path: Path, output_dir: Path, chunk_sec: float = 8.0) -> list[Path]:
    info = probe_video(video_path)
    duration = info["duration"]
    chunks = []
    start = 0.0
    idx = 0
    while start < duration:
        end = min(start + chunk_sec, duration)
        out_path = output_dir / f"chunk_{idx:04d}.mp4"
        cmd = [
            "ffmpeg", "-y", "-i", str(video_path),
            "-ss", str(start), "-to", str(end),
            "-c", "copy", "-avoid_negative_ts", "1",
            str(out_path)
        ]
        subprocess.run(cmd, capture_output=True)
        if out_path.exists() and out_path.stat().st_size > 0:
            chunks.append(out_path)
        start = end
        idx += 1
    return chunks

def extract_frames(video_path: Path, fps: float = None) -> list[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    return frames

def frames_to_video(frames: list[np.ndarray], output_path: Path, fps: float, audio_path: Path = None):
    if not frames:
        return
    h, w = frames[0].shape[:2]
    tmp = output_path.with_suffix(".tmp.mp4")
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(tmp), fourcc, fps, (w, h))
    for f in frames:
        writer.write(f)
    writer.release()
    if audio_path and audio_path.exists():
        cmd = ["ffmpeg", "-y", "-i", str(tmp), "-i", str(audio_path),
               "-c:v", "copy", "-c:a", "aac", "-shortest", str(output_path)]
        subprocess.run(cmd, capture_output=True)
        tmp.unlink(missing_ok=True)
    else:
        shutil.move(str(tmp), str(output_path))

def stitch_videos(chunk_paths: list[Path], output_path: Path, audio_path: Path = None):
    concat_file = output_path.parent / "concat_list.txt"
    with open(concat_file, "w") as f:
        for p in chunk_paths:
            f.write(f"file \'{p.resolve()}\'\\n")
    cmd = ["ffmpeg", "-y", "-f", "concat", "-safe", "0",
           "-i", str(concat_file), "-c", "copy", str(output_path)]
    subprocess.run(cmd, capture_output=True)
    concat_file.unlink(missing_ok=True)

# ---- Pipeline Components (lazy loaded) ----
_pipeline_cache = {}

def get_face_detector():
    if "face_det" not in _pipeline_cache:
        try:
            from insightface.app import FaceAnalysis
            app = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider"])
            app.prepare(ctx_id=0, det_size=(640, 640))
            _pipeline_cache["face_det"] = app
            logger.info("✅ InsightFace loaded")
        except Exception as e:
            logger.warning(f"InsightFace not available: {e}")
            _pipeline_cache["face_det"] = None
    return _pipeline_cache["face_det"]

def get_controlnet_pipeline():
    if "pipe" not in _pipeline_cache:
        logger.info("Loading ControlNet + SD pipeline... (first run takes 2-3 min)")
        try:
            from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, DDIMScheduler
            from controlnet_aux import OpenposeDetector

            controlnet = ControlNetModel.from_pretrained(
                "lllyasviel/control_v11p_sd15_openpose",
                torch_dtype=torch.float16
            )
            pipe = StableDiffusionControlNetPipeline.from_pretrained(
                "runwayml/stable-diffusion-v1-5",
                controlnet=controlnet,
                torch_dtype=torch.float16,
                safety_checker=None,
            ).to("cuda")
            pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
            pipe.enable_vae_slicing()

            pose_detector = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")

            _pipeline_cache["pipe"] = pipe
            _pipeline_cache["pose"] = pose_detector
            logger.info("✅ ControlNet pipeline loaded")
        except Exception as e:
            logger.error(f"Failed to load pipeline: {e}")
            _pipeline_cache["pipe"] = None
            _pipeline_cache["pose"] = None
    return _pipeline_cache.get("pipe"), _pipeline_cache.get("pose")

# ---- Processing Functions ----
def process_frame_swap(frame: np.ndarray, reference_img: np.ndarray, pipe, pose_detector) -> np.ndarray:
    """Process a single frame through the swap pipeline."""
    try:
        h, w = frame.shape[:2]
        # Compute latent dims (multiple of 8)
        max_dim = 512
        scale = max_dim / max(w, h)
        lat_w = int(round(w * scale / 8) * 8)
        lat_h = int(round(h * scale / 8) * 8)

        # Extract pose
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_frame = Image.fromarray(frame_rgb).resize((lat_w, lat_h))
        pose_image = pose_detector(pil_frame)

        # Generate with identity prompt
        result = pipe(
            prompt="photorealistic portrait, high quality, sharp focus, natural skin",
            negative_prompt="lowres, bad anatomy, worst quality, blurry, deformed",
            image=pose_image,
            num_inference_steps=20,
            guidance_scale=7.5,
            width=lat_w,
            height=lat_h,
        ).images[0]

        # Resize back
        result_bgr = cv2.cvtColor(np.array(result), cv2.COLOR_RGB2BGR)
        result_resized = cv2.resize(result_bgr, (w, h), interpolation=cv2.INTER_LANCZOS4)

        # Blend with original (artifact removal)
        alpha = 0.85
        blended = cv2.addWeighted(result_resized, alpha, frame, 1.0 - alpha, 0)
        return blended

    except Exception as e:
        logger.error(f"Frame processing error: {e}")
        return frame  # Return original on failure

async def process_job_background(job_id: str):
    """Background task that processes the entire video swap job."""
    job = jobs.get(job_id)
    if not job:
        return

    try:
        jobs[job_id]["status"] = JobStatus.PROCESSING
        source_path = Path(job["source_path"])
        ref_path = Path(job["reference_path"])
        job_dir = UPLOAD_DIR / job_id

        # Load reference image
        ref_img = cv2.imread(str(ref_path))
        if ref_img is None:
            raise ValueError("Could not read reference image")

        # Probe video
        info = probe_video(source_path)
        fps = info["fps"]
        logger.info(f"[{job_id}] Video: {info['width']}x{info['height']} @ {fps:.1f}fps, {info['duration']:.1f}s")

        # Chunk video
        jobs[job_id]["status"] = JobStatus.CHUNKING
        chunk_dir = job_dir / "chunks"
        chunk_dir.mkdir(exist_ok=True)
        chunks = chunk_video(source_path, chunk_dir)
        total_chunks = len(chunks)
        logger.info(f"[{job_id}] Split into {total_chunks} chunks")

        # Load pipeline
        jobs[job_id]["status"] = JobStatus.PROCESSING
        pipe, pose_detector = get_controlnet_pipeline()

        # Process each chunk
        processed_chunks = []
        for ci, chunk_path in enumerate(chunks):
            logger.info(f"[{job_id}] Processing chunk {ci+1}/{total_chunks}")

            # Extract frames
            frames = extract_frames(chunk_path)
            processed_frames = []

            for fi, frame in enumerate(frames):
                if pipe and pose_detector:
                    result = process_frame_swap(frame, ref_img, pipe, pose_detector)
                else:
                    result = frame  # Pass-through if pipeline not loaded
                processed_frames.append(result)

                if (fi + 1) % 10 == 0:
                    overall = ((ci * len(frames) + fi + 1) / (total_chunks * len(frames))) * 100
                    jobs[job_id]["progress"] = round(overall, 1)
                    logger.info(f"[{job_id}] Frame {fi+1}/{len(frames)} in chunk {ci+1} ({overall:.1f}% overall)")

            # Extract audio from original chunk
            audio_path = chunk_path.with_suffix(".wav")
            subprocess.run(
                ["ffmpeg", "-y", "-i", str(chunk_path), "-vn", "-acodec", "pcm_s16le", str(audio_path)],
                capture_output=True
            )

            # Write processed chunk
            proc_path = PROCESSED_DIR / job_id / f"processed_chunk_{ci:04d}.mp4"
            proc_path.parent.mkdir(parents=True, exist_ok=True)
            frames_to_video(
                processed_frames, proc_path, fps,
                audio_path if audio_path.exists() else None
            )
            processed_chunks.append(proc_path)

            # Clear VRAM between chunks
            clear_vram()
            logger.info(f"[{job_id}] Chunk {ci+1}/{total_chunks} complete")

        # Stitch all chunks
        jobs[job_id]["status"] = JobStatus.STITCHING
        output_path = OUTPUT_DIR / f"{job_id}_final.mp4"
        stitch_videos(processed_chunks, output_path)

        jobs[job_id]["status"] = JobStatus.COMPLETED
        jobs[job_id]["progress"] = 100.0
        jobs[job_id]["output_path"] = str(output_path)
        logger.info(f"[{job_id}] ✅ JOB COMPLETED! Output: {output_path}")

    except Exception as e:
        logger.error(f"[{job_id}] ❌ Job failed: {e}")
        jobs[job_id]["status"] = JobStatus.FAILED
        jobs[job_id]["error"] = str(e)
    finally:
        clear_vram()


# ---- FastAPI App ----
app = FastAPI(title="Video Swap Pipeline API", version="1.0.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health")
async def health():
    return {
        "status": "ok",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
        "vram_free_gb": round(torch.cuda.mem_get_info(0)[0] / (1024**3), 2) if torch.cuda.is_available() else 0,
    }

@app.post("/api/v1/jobs/create")
async def create_job(
    background_tasks: BackgroundTasks,
    source_video: UploadFile = File(...),
    reference_image: UploadFile = File(...),
):
    job_id = str(uuid.uuid4())
    job_dir = UPLOAD_DIR / job_id
    job_dir.mkdir(parents=True, exist_ok=True)

    # Save files
    src_path = job_dir / f"source{Path(source_video.filename or '.mp4').suffix}"
    ref_path = job_dir / f"reference{Path(reference_image.filename or '.png').suffix}"

    with open(src_path, "wb") as f:
        while chunk := await source_video.read(1024 * 1024):
            f.write(chunk)

    with open(ref_path, "wb") as f:
        while chunk := await reference_image.read(1024 * 1024):
            f.write(chunk)

    # Check duration
    info = probe_video(src_path)
    if info["duration"] > MAX_DURATION:
        shutil.rmtree(job_dir)
        raise HTTPException(413, f"Video too long ({info['duration']:.0f}s > {MAX_DURATION}s max)")

    jobs[job_id] = {
        "status": JobStatus.PENDING,
        "progress": 0.0,
        "source_path": str(src_path),
        "reference_path": str(ref_path),
        "output_path": None,
        "error": None,
    }

    # Start processing in background
    background_tasks.add_task(process_job_background, job_id)

    return {
        "job_id": job_id,
        "status": "PENDING",
        "message": f"Job created. Video: {info['width']}x{info['height']}, {info['duration']:.1f}s",
        "duration": info["duration"],
    }

@app.get("/api/v1/jobs/{job_id}/status")
async def get_status(job_id: str):
    if job_id not in jobs:
        raise HTTPException(404, "Job not found")
    j = jobs[job_id]
    return {
        "job_id": job_id,
        "status": j["status"],
        "progress": j["progress"],
        "error": j.get("error"),
    }

@app.get("/api/v1/jobs/{job_id}/download")
async def download(job_id: str):
    if job_id not in jobs:
        raise HTTPException(404, "Job not found")
    j = jobs[job_id]
    if j["status"] != JobStatus.COMPLETED or not j.get("output_path"):
        raise HTTPException(400, f"Job status: {j['status']}. Not ready yet.")
    return FileResponse(j["output_path"], media_type="video/mp4", filename=f"swapped_{job_id}.mp4")

@app.delete("/api/v1/jobs/{job_id}")
async def cancel(job_id: str):
    if job_id in jobs:
        jobs[job_id]["status"] = JobStatus.FAILED
    return {"message": f"Job {job_id} cancelled"}
'''

server_path = PROJECT_DIR / "server.py"
with open(server_path, "w") as f:
    f.write(MINIMAL_SERVER)

print(f"✅ Server code written to {server_path}")
print("✅ Proceed to Cell 4.")

## Cell 4: Pre-load AI Models (One Time)

In [ ]:
# ============================================================
# CELL 4: Pre-download & cache the AI models
# First run: ~5-8 minutes (downloads ~4GB of model weights)
# Subsequent runs: instant (cached)
# ============================================================
import torch
print("🔄 Pre-loading AI models (first time takes 5-8 min)...")
print("=" * 60)

# 1. ControlNet model
print("\n📥 [1/3] Downloading ControlNet OpenPose...")
try:
    from diffusers import ControlNetModel
    controlnet = ControlNetModel.from_pretrained(
        "lllyasviel/control_v11p_sd15_openpose",
        torch_dtype=torch.float16
    )
    del controlnet
    print("   ✅ ControlNet cached")
except Exception as e:
    print(f"   ⚠️ ControlNet download issue: {e}")

# 2. Stable Diffusion base model
print("\n📥 [2/3] Downloading Stable Diffusion v1.5...")
try:
    from diffusers import StableDiffusionControlNetPipeline
    pipe = StableDiffusionControlNetPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        torch_dtype=torch.float16,
        safety_checker=None,
    )
    del pipe
    print("   ✅ SD 1.5 cached")
except Exception as e:
    print(f"   ⚠️ SD download issue: {e}")

# 3. OpenPose detector
print("\n📥 [3/3] Downloading OpenPose detector...")
try:
    from controlnet_aux import OpenposeDetector
    pose = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")
    del pose
    print("   ✅ OpenPose cached")
except Exception as e:
    print(f"   ⚠️ OpenPose download issue: {e}")

# 4. InsightFace
print("\n📥 [Bonus] Checking InsightFace...")
try:
    from insightface.app import FaceAnalysis
    fa = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider"])
    fa.prepare(ctx_id=0, det_size=(640, 640))
    del fa
    print("   ✅ InsightFace ready")
except Exception as e:
    print(f"   ⚠️ InsightFace: {e} (face detection will use fallback)")

# Clear VRAM after downloads
torch.cuda.empty_cache()
import gc; gc.collect()

free_vram = torch.cuda.mem_get_info(0)[0] / (1024**3)
print(f"\n✅ All models cached! Free VRAM: {free_vram:.1f} GB")
print("✅ Proceed to Cell 5 to start the server.")

## Cell 5: 🚀 Start Server & Get Public URL

In [ ]:
# ============================================================
# CELL 5: Start the FastAPI server and expose via ngrok tunnel
# ============================================================
import subprocess
import time
import threading

# Start uvicorn in the background
print("🚀 Starting FastAPI server...")
server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/video-swap-pipeline",
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(3)

# Check it's running
if server_proc.poll() is None:
    print("✅ Server running on port 8000")
else:
    print("❌ Server failed to start!")
    print(server_proc.stderr.read().decode())

# Create ngrok tunnel
print("\n🌐 Creating public tunnel...")
print("=" * 60)

try:
    from pyngrok import ngrok

    # If you have an ngrok auth token, uncomment and set it:
    # ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")

    tunnel = ngrok.connect(8000, "http")
    public_url = tunnel.public_url

    print(f"\n{'='*60}")
    print(f"🎉 YOUR PUBLIC API URL:")
    print(f"")
    print(f"   {public_url}")
    print(f"")
    print(f"📋 Copy this URL into your Android app settings!")
    print(f"{'='*60}")
    print(f"")
    print(f"📡 API Endpoints:")
    print(f"   Health Check:  GET  {public_url}/health")
    print(f"   Create Job:    POST {public_url}/api/v1/jobs/create")
    print(f"   Job Status:    GET  {public_url}/api/v1/jobs/{{job_id}}/status")
    print(f"   Download:      GET  {public_url}/api/v1/jobs/{{job_id}}/download")
    print(f"   Cancel Job:    DEL  {public_url}/api/v1/jobs/{{job_id}}")
    print(f"   API Docs:      GET  {public_url}/docs")
    print(f"")
    print(f"⏰ This session lasts ~4 hours. Keep this tab open!")

except Exception as e:
    print(f"\n⚠️ ngrok tunnel failed: {e}")
    print(f"\nFallback: Use Colab's built-in proxy:")
    print(f"  Your API is at: http://localhost:8000")
    print(f"  For external access, get a free ngrok token at https://ngrok.com")

## Cell 6: 🧪 Test the API

In [ ]:
# ============================================================
# CELL 6: Quick test of the API
# ============================================================
import requests

# Test health endpoint
print("Testing health endpoint...")
resp = requests.get("http://localhost:8000/health")
print(f"Status: {resp.status_code}")
print(f"Response: {resp.json()}")

print("\n✅ Server is live and ready for video swap jobs!")
print("\n📱 To test from your phone:")
print("1. Open a browser on your Itel S26 Ultra")
print("2. Navigate to: <your_ngrok_url>/docs")
print("3. Use the Swagger UI to upload a test video + reference image")
print("4. Or use the Android app (Gemini's part) pointed at the ngrok URL")

## Cell 7: 📊 Monitor Running Jobs

In [ ]:
# ============================================================
# CELL 7: Monitor GPU usage and running jobs
# Run this cell anytime to check status
# ============================================================
import torch
import requests

print("=" * 60)
print("GPU STATUS")
print("=" * 60)
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv,noheader

print("\n" + "=" * 60)
print("SERVER HEALTH")
print("=" * 60)
try:
    resp = requests.get("http://localhost:8000/health", timeout=5)
    print(f"Server: {'🟢 Online' if resp.ok else '🔴 Error'}")
    print(f"Details: {resp.json()}")
except:
    print("🔴 Server not responding")

---
## ⚠️ Important Notes

- **Keep this tab open** — Colab disconnects if you close it
- **Free tier limit**: ~4 hours of GPU time per session
- **If disconnected**: Just run all cells again (models are cached, takes ~1 min)
- **For longer sessions**: Consider [Colab Pro](https://colab.research.google.com/signup) ($10/month)
- **Processing speed**: ~2-5 seconds per frame on T4 GPU
  - 10-second video (300 frames @ 30fps) ≈ 10-25 minutes
  - 1-minute video ≈ 1-2.5 hours
  - 10-minute video ≈ 10-25 hours (may exceed session limit!)